# Brunata Online — API Debug Notebook
Tests the Brunata API client directly without Home Assistant.
Set credentials via environment variables or edit the cell below.

In [ ]:
import sys, os

# Ensure the project root is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..' if os.path.basename(os.getcwd()) != 'ha-brunata' else '.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('sys.path includes:', PROJECT_ROOT)

In [ ]:
# Credentials — set BRUNATA_USERNAME and BRUNATA_PASSWORD env vars,
# or replace the os.getenv() calls with your actual values.
USERNAME = os.getenv('BRUNATA_USERNAME', 'your@email.com')
PASSWORD = os.getenv('BRUNATA_PASSWORD', 'your_password')
print(f'Using username: {USERNAME}')

In [ ]:
import aiohttp
import datetime
from aiohttp import ClientSession, CookieJar, TraceConfig

# Optional request tracing — set to True to see raw HTTP headers
TRACE = False
# Optional HTTP proxy (e.g. Burp Suite / mitmproxy) — set to None to disable
PROXY = None  # e.g. 'http://127.0.0.1:8080'

trace_configs = []
if TRACE:
    async def _on_request_start(session, ctx, params):
        print(f'>>> {params.method} {params.url}')
    async def _on_request_headers_sent(session, ctx, params):
        for k, v in params.headers.items():
            print(f'  {k}: {v}')
    tc = TraceConfig()
    tc.on_request_start.append(_on_request_start)
    tc.on_request_headers_sent.append(_on_request_headers_sent)
    trace_configs = [tc]

connector = aiohttp.TCPConnector(ssl=False) if PROXY else None
session = ClientSession(
    cookie_jar=CookieJar(unsafe=True, quote_cookie=False),
    trace_configs=trace_configs,
    **(dict(proxy=PROXY, connector=connector) if PROXY else {})
)
print('Session created')

In [ ]:
from custom_components.brunata_online.api.brunata_api.client import BrunataClient

client = BrunataClient(USERNAME, PASSWORD, session, 'en')
print('Client created')

## 1. Authentication

In [ ]:
# Trigger authentication by fetching allocation units (first call that requires a token)
allocation_units = await client._get_allocation_units()
print('Allocation units:', allocation_units)

## 2. Meters

In [ ]:
meters = await client.get_meters()
for m in meters:
    print(f'  [{m.meter_type_code}] {m.meter_type} — {m.value_category} ({m.unit}) | id={m.meter_id} alloc={m.allocation_unit_code}')

## 3. Consumption data

In [ ]:
from custom_components.brunata_online.api.const import Consumption, Interval

end_time = datetime.datetime.today()
start_time = end_time - datetime.timedelta(days=30)

for meter in meters:
    try:
        ct = Consumption(meter.meter_type_code)
    except ValueError:
        print(f'Unknown type code {meter.meter_type_code}, skipping')
        continue
    readings = await client.get_consumption(start_time, end_time, ct, meter.allocation_unit_code, Interval.DAY)
    print(f'\n--- {meter.value_category} (meter {meter.meter_id}) ---')
    for r in readings:
        for v in r.Values:
            print(f'  {v.fromDate.date()}  {v.consumption}')

In [ ]:
await session.close()
print('Session closed')